In [2]:
import pandas as pd
import openai
import logging
import time
import os
import json  # For serialization
from tqdm import tqdm
from datetime import datetime
from secret_keys import OPENAI_API_KEY  # Import your OpenAI API key from a separate file

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("extract_pubmed.log"),
        logging.StreamHandler()
    ]
)

# Load your OpenAI API key
openai.api_key = OPENAI_API_KEY

def get_embeddings(texts, model="text-embedding-3-small", max_retries=5, backoff_factor=2):
    """
    Retrieve embeddings for a list of texts using OpenAI's Embedding API.
    
    Args:
        texts (list): A list of strings to embed.
        model (str): The embedding model to use.
        max_retries (int): Maximum number of retries for failed API calls.
        backoff_factor (int): Factor by which the wait time increases after each retry.
        
    Returns:
        list: A list of embeddings corresponding to the input texts.
    """
    for attempt in range(max_retries):
        try:
            response = openai.Embedding.create(input=texts, model=model)
            embeddings = [data['embedding'] for data in response['data']]
            return embeddings
        except openai.error.RateLimitError:
            wait = backoff_factor ** attempt
            logging.warning(f"Rate limit exceeded. Retrying in {wait} seconds... (Attempt {attempt + 1}/{max_retries})")
            time.sleep(wait)
        except openai.error.OpenAIError as e:
            logging.error(f"OpenAI API error: {e}")
            break
        except Exception as e:
            logging.error(f"Unexpected error: {e}")
            break
    logging.error("Failed to retrieve embeddings after multiple attempts.")
    return [None] * len(texts)  # Return None embeddings for failed attempts

def add_embeddings_to_records(csv_file_path, output_file_path, batch_size=100):
    """
    Read a CSV file, generate embeddings for the 'Abstract' column where embeddings are missing, and save the results incrementally.
    
    Args:
        csv_file_path (str): Path to the input CSV file containing PubMed records.
        output_file_path (str): Path to the output CSV file to save embeddings.
        batch_size (int): Number of records to process in each batch.
    """
    # Check if output file exists
    if os.path.exists(output_file_path):
        # Resume processing from the output file
        logging.info(f"Resuming from existing output file: {output_file_path}")
        processed_ids = set()
        # Read the output file to get IDs of already processed records
        try:
            for chunk in pd.read_csv(output_file_path, usecols=['PMID'], chunksize=batch_size):
                processed_ids.update(chunk['PMID'].tolist())
        except Exception as e:
            logging.error(f"Error reading output file for processed IDs: {e}")
            return
        write_header = False  # Don't write header if appending to existing file
    else:
        processed_ids = set()
        write_header = True  # Write header if creating a new file

    # Read the input CSV file in chunks
    try:
        df_iter = pd.read_csv(csv_file_path, chunksize=batch_size)
    except FileNotFoundError:
        logging.error(f"Input file {csv_file_path} not found.")
        return
    except pd.errors.EmptyDataError:
        logging.error(f"Input file {csv_file_path} is empty.")
        return
    except Exception as e:
        logging.error(f"Error reading input file: {e}")
        return

    # Initialize progress bar
    try:
        with open(csv_file_path, 'r', encoding='utf-8') as f:
            total_records = sum(1 for _ in f) - 1  # Subtract header
    except Exception as e:
        logging.warning(f"Could not determine total records: {e}")
        total_records = None

    pbar = tqdm(total=total_records, desc="Processing Abstracts") if total_records else tqdm(desc="Processing Abstracts")

    for chunk_number, chunk in enumerate(df_iter, start=1):
        # Ensure 'Abstract' and 'PMID' columns exist
        if 'Abstract' not in chunk.columns or 'PMID' not in chunk.columns:
            logging.error("The required columns ('Abstract', 'PMID') are missing in the input file.")
            return

        # Filter out records that have already been processed
        unprocessed_chunk = chunk[~chunk['PMID'].isin(processed_ids)].copy()

        if unprocessed_chunk.empty:
            # If all records in this chunk have been processed, skip to the next chunk
            pbar.update(len(chunk))
            continue

        # Initialize 'Embedding' column if it doesn't exist
        if 'Embedding' not in unprocessed_chunk.columns:
            unprocessed_chunk['Embedding'] = None

        # Ensure 'Abstract' column is of string type
        unprocessed_chunk['Abstract'] = unprocessed_chunk['Abstract'].astype(str)

        # Handle missing or empty abstracts
        mask = unprocessed_chunk['Abstract'].notnull() & (unprocessed_chunk['Abstract'].str.strip() != "")
        abstracts = unprocessed_chunk.loc[mask, 'Abstract'].tolist()

        if abstracts:
            # Retrieve embeddings for the batch
            embeddings = get_embeddings(abstracts)

            # Check if embeddings length matches abstracts length
            if len(embeddings) != len(abstracts):
                logging.error(f"Mismatch in number of embeddings received: Expected {len(abstracts)}, Got {len(embeddings)}")
                # Assign None to all for this batch
                unprocessed_chunk.loc[mask, 'Embedding'] = None
            else:
                # Serialize each embedding as JSON string
                embeddings_serialized = [json.dumps(e) if e is not None else None for e in embeddings]
                # Assign embeddings to the corresponding rows
                unprocessed_chunk.loc[mask, 'Embedding'] = embeddings_serialized
        else:
            # If all abstracts are empty, assign None
            unprocessed_chunk['Embedding'] = None

        # Append unprocessed_chunk to the output CSV
        try:
            unprocessed_chunk.to_csv(output_file_path, mode='a', index=False, header=write_header)
            if write_header:
                write_header = False  # Header written once
        except Exception as e:
            logging.error(f"Error writing to output file: {e}")
            return

        # Update processed IDs
        processed_ids.update(unprocessed_chunk['PMID'].tolist())

        # Update progress bar
        pbar.update(len(chunk))
        logging.info(f"Processed chunk {chunk_number} with {len(unprocessed_chunk)} new records.")

    pbar.close()
    logging.info("Embedding process completed successfully.")

# Example usage
if __name__ == "__main__":
    input_csv = "pubmed_results_complete.csv"              # Replace with your input CSV file path
    output_csv = "pubmed_results_with_embeddings.csv"      # Replace with your desired output CSV file path
    batch_size = 100                                       # Adjust based on your requirements and API limits

    # Ensure 'PMID' column exists
    if not os.path.exists(input_csv):
        logging.error(f"Input file {input_csv} not found.")
    else:
        first_chunk = pd.read_csv(input_csv, nrows=1)
        if 'PMID' not in first_chunk.columns:
            logging.error("The 'PMID' column is missing in the input file. Please ensure your input CSV contains 'PMID'.")
            exit(1)

    add_embeddings_to_records(
        csv_file_path=input_csv,
        output_file_path=output_csv,
        batch_size=batch_size
    )


Processing Abstracts:  23%|██▎       | 500/2198 [00:14<00:48, 35.24it/s]2025-09-04 16:12:49,286 - INFO - Processed chunk 5 with 100 new records.


KeyboardInterrupt: 

Processing Abstracts:  23%|██▎       | 500/2198 [00:30<00:48, 35.24it/s]